# Objectives

- Use linear regression in one variable to fit the parameters to a model
- Use linear regression in multiple variables to fit the parameters to a model
- Use polynomial regression in a single variable to fit the parameters to a model
- Create a pipeline for performing linear regression using multiple features in polynomial scaling
- Use the grid search with cross-validation and ridge regression to create a model with optimum hyperparameters

# Install required libraries

In [1]:
%pip install seaborn
import piplite

await piplite.install(['nbformat', 'plotly'])

# Dataset URL from the GenAI lab

In [2]:
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DA0101EN-Coursera/laptop_pricing_dataset_mod2.csv"

In [3]:
from pyodide.http import pyfetch

async def download(url, filename):
    response = await pyfetch(url)
    if response.status == 200:
        with open(filename, "wb") as f:
            f.write(await response.bytes())

path = URL

await download(path, "data2.csv")
file_name  = "data2.csv"

# Test environment

In [5]:
import pandas as pd

file_path = 'data2.csv'
df = pd.read_csv(file_path, header=0)

In [9]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Configuration: adjust these for your data
file_path = "data2.csv"  # CSV with header row
feature_col = "CPU_frequency"  # source variable (X)
target_col = "Price"    # target variable (y)

# Load data
df = pd.read_csv(file_path, header=0)

# Prepare features and target
X = df[[feature_col]]
y = df[target_col]

# Train model
model = LinearRegression()
model.fit(X, y)

# Predict and evaluate
y_pred = model.predict(X)
mse = mean_squared_error(y, y_pred)
r2 = r2_score(y, y_pred)

# Output results
print("MSE:", mse)
print("R^2:", r2)

MSE: 284583.44058686297
R^2: 0.13444363210243238


In [10]:
# replace with actual feature columns and target
feature_columns = ['CPU_frequency', 'RAM_GB', 'Storage_GB_SSD', 'CPU_core', 'OS', 'GPU', 'Category']  # some attributes
target_column = 'Price'                   # one attribute as target

X = df[feature_columns]
y = df[target_column]

model = LinearRegression().fit(X, y)
pred = model.predict(X)

mse = mean_squared_error(y, pred)
r2 = r2_score(y, pred)

print('MSE:', mse)
print('R^2:', r2)

MSE: 161680.57263893107
R^2: 0.5082509055187374


In [11]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# replace with actual column names
X = df[['CPU_frequency']]
y = df['Price']

degrees = [2, 3, 5]
results = []

for d in degrees:
    model = Pipeline([
        ('poly', PolynomialFeatures(degree=d, include_bias=False)),
        ('lr', LinearRegression())
    ])
    model.fit(X, y)
    pred = model.predict(X)
    mse = mean_squared_error(y, pred)
    r2 = r2_score(y, pred)
    results.append((d, mse, r2))

for d, mse, r2 in results:
    print('Degree', d, 'MSE:', mse, 'R^2:', r2)

best_by_r2 = max(results, key=lambda t: t[2])
best_by_mse = min(results, key=lambda t: t[1])

print('Best by R^2: degree', best_by_r2[0], 'R^2', best_by_r2[2])
print('Best by MSE: degree', best_by_mse[0], 'MSE', best_by_mse[1])

Degree 2 MSE: 249022.66596751165 R^2: 0.24260120745423797
Degree 3 MSE: 241024.86303848762 R^2: 0.2669264079653114
Degree 5 MSE: 229137.29548054087 R^2: 0.30308227064437243
Best by R^2: degree 5 R^2 0.30308227064437243
Best by MSE: degree 5 MSE 229137.29548054087


In [12]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# replace with actual feature columns and target
feature_columns = ['CPU_frequency', 'RAM_GB', 'Storage_GB_SSD', 'CPU_core', 'OS', 'GPU', 'Category']  # multiple features
target_column = 'Price'                   # target variable

X = df[feature_columns]
y = df[target_column]

degree = 2  # degree for polynomial features

model = Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
    ('lr', LinearRegression())
])

model.fit(X, y)
pred = model.predict(X)

mse = mean_squared_error(y, pred)
r2 = r2_score(y, pred)

print('MSE:', mse)
print('R^2:', r2)

MSE: 244507.9894957983
R^2: 0.2563325298426047


In [13]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

# replace with actual feature columns and target
feature_columns = ['CPU_frequency', 'RAM_GB', 'Storage_GB_SSD', 'CPU_core', 'OS', 'GPU', 'Category']  # some attributes
target_column = 'Price'                   # target variable

X = df[feature_columns]
y = df[target_column]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(include_bias=False)),
    ('ridge', Ridge())
])

# param_grid = {
#    'poly__degree': [2, 3, 5],
#    'ridge__alpha': [0.1, 1.0, 10.0, 100.0]
# }

param_grid = {
    "preprocessor__poly__degree": [2, 3],
    "model__alpha": [0.1, 1.0, 10.0]
}

grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='neg_mean_squared_error')
grid.fit(X_train, y_train)

best = grid.best_estimator_
pred_test = best.predict(X_test)

mse = mean_squared_error(y_test, pred_test)
r2 = r2_score(y_test, pred_test)

print('Best params:', grid.best_params_)
print('Test MSE:', mse)
print('Test R^2:', r2)

<class 'ValueError'>: Invalid parameter 'model' for estimator Pipeline(steps=[('scaler', StandardScaler()),
                ('poly', PolynomialFeatures(include_bias=False)),
                ('ridge', Ridge())]). Valid parameters are: ['memory', 'steps', 'verbose'].